# LexiScreen — Level 2 Machine Learning Risk Modeling
### Multimodal Pediatric Learning Difficulty Screening Pipeline (2026)

This notebook trains, evaluates, and exports the **Level 2 Machine Learning Statistical Risk Model** for the **LexiScreen Multimodal Learning Difficulty Screening System**.

**Team Role**: Backend Member 2 (Feature extraction, Level 1 Scoring & Level 2 ML Pipeline)

---

## 1. Environment Setup & Imports
No external GPU or paid tier required. Runs on free Google Colab CPU.

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve, auc
)
import joblib

print("All libraries successfully imported!")

## 2. Synthetic Multimodal Screening Dataset Generation
Generates realistic pediatric screening cohort distributions across grades K-6 covering handwriting kinematics, oral speech pauses/WPM, and letter reversals.

In [ ]:
def generate_multimodal_dataset(n_samples=1500, random_seed=42):
    np.random.seed(random_seed)
    classes = np.random.choice([0, 1, 2], size=n_samples, p=[0.55, 0.25, 0.20])
    grades = ['K', '1', '2', '3', '4', '5', '6']
    grade_targets = {'K': 35, '1': 55, '2': 85, '3': 110, '4': 130, '5': 145, '6': 160}
    
    data = []
    for i, c in enumerate(classes):
        grade = np.random.choice(grades)
        target_wpm = grade_targets[grade]
        grade_num = grades.index(grade)
        
        if c == 0:  # Low Risk
            reversal_rate = np.clip(np.random.normal(0.04, 0.04), 0.0, 0.20)
            wpm_deficit = np.clip(np.random.normal(-0.10, 0.15), -0.50, 0.15)
            decoding_acc = np.clip(np.random.normal(0.95, 0.04), 0.85, 1.0)
            pause_count = max(0, int(np.random.normal(1.2, 1.0)))
            pause_duration_ms = max(200, int(np.random.normal(1500, 800)))
            silence_ratio = np.clip(np.random.normal(0.10, 0.05), 0.02, 0.25)
            avg_hesitation_ms = max(300, int(np.random.normal(800, 300)))
            hw_jitter = np.clip(np.random.normal(0.35, 0.12), 0.15, 0.70)
            hw_pen_lifts = max(0, int(np.random.normal(1.2, 0.8)))
            hw_velocity_cv = np.clip(np.random.normal(0.40, 0.12), 0.20, 0.70)
            hw_consistency = np.clip(int(np.random.normal(88, 7)), 70, 100)
        elif c == 1:  # Moderate Risk
            reversal_rate = np.clip(np.random.normal(0.28, 0.10), 0.12, 0.50)
            wpm_deficit = np.clip(np.random.normal(0.30, 0.15), 0.10, 0.60)
            decoding_acc = np.clip(np.random.normal(0.82, 0.06), 0.68, 0.92)
            pause_count = max(1, int(np.random.normal(4.0, 1.5)))
            pause_duration_ms = max(1000, int(np.random.normal(5500, 1800)))
            silence_ratio = np.clip(np.random.normal(0.28, 0.08), 0.15, 0.45)
            avg_hesitation_ms = max(800, int(np.random.normal(1800, 500)))
            hw_jitter = np.clip(np.random.normal(0.75, 0.18), 0.45, 1.10)
            hw_pen_lifts = max(1, int(np.random.normal(3.0, 1.2)))
            hw_velocity_cv = np.clip(np.random.normal(0.75, 0.15), 0.50, 1.05)
            hw_consistency = np.clip(int(np.random.normal(68, 8)), 50, 82)
        else:  # High Risk
            reversal_rate = np.clip(np.random.normal(0.65, 0.15), 0.40, 1.0)
            wpm_deficit = np.clip(np.random.normal(0.65, 0.18), 0.35, 1.0)
            decoding_acc = np.clip(np.random.normal(0.62, 0.10), 0.35, 0.78)
            pause_count = max(3, int(np.random.normal(8.5, 2.5)))
            pause_duration_ms = max(3000, int(np.random.normal(14000, 4500)))
            silence_ratio = np.clip(np.random.normal(0.50, 0.12), 0.30, 0.85)
            avg_hesitation_ms = max(1500, int(np.random.normal(3200, 800)))
            hw_jitter = np.clip(np.random.normal(1.20, 0.25), 0.80, 1.80)
            hw_pen_lifts = max(2, int(np.random.normal(5.5, 1.8)))
            hw_velocity_cv = np.clip(np.random.normal(1.10, 0.20), 0.80, 1.60)
            hw_consistency = np.clip(int(np.random.normal(42, 10)), 15, 60)
            
        data.append({
            'reversal_error_rate': reversal_rate,
            'wpm_deficit_ratio': wpm_deficit,
            'decoding_accuracy_pct': decoding_acc * 100,
            'pause_count': pause_count,
            'silence_ratio': silence_ratio,
            'avg_hesitation_ms': avg_hesitation_ms,
            'handwriting_jitter': hw_jitter,
            'handwriting_pen_lifts': hw_pen_lifts,
            'handwriting_velocity_cv': hw_velocity_cv,
            'handwriting_consistency': hw_consistency,
            'risk_class': int(c)
        })
        
    return pd.DataFrame(data)

df = generate_multimodal_dataset(1500)
print(f"Generated dataset: {df.shape}")
df.head()

## 3. Data Splitting & Feature Scaling

In [ ]:
FEATURE_COLS = [
    'reversal_error_rate',
    'wpm_deficit_ratio',
    'decoding_accuracy_pct',
    'pause_count',
    'silence_ratio',
    'avg_hesitation_ms',
    'handwriting_jitter',
    'handwriting_pen_lifts',
    'handwriting_velocity_cv',
    'handwriting_consistency'
]
TARGET_COL = 'risk_class'

X = df[FEATURE_COLS]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

## 4. Train Models: Logistic Regression, Decision Tree & Random Forest

In [ ]:
# 1. Logistic Regression
lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)

# 2. Decision Tree
dt = DecisionTreeClassifier(max_depth=5, min_samples_split=10, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)

# 3. Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)

print("Model training complete!")

## 5. Evaluation & Comparison Metrics

In [ ]:
def evaluate(name, y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    roc = roc_auc_score(y_true, y_prob, multi_class='ovr')
    print(f"=== {name} ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"F1-Macro:  {f1:.4f}")
    print(f"ROC-AUC:   {roc:.4f}")
    print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=['Low', 'Moderate', 'High']))
    return {'Accuracy': acc, 'F1-Score': f1, 'ROC-AUC': roc}

res_lr = evaluate("Logistic Regression", y_test, y_pred_lr, y_prob_lr)
res_dt = evaluate("Decision Tree", y_test, y_pred_dt, y_prob_dt)
res_rf = evaluate("Random Forest", y_test, y_pred_rf, y_prob_rf)

## 6. Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Low', 'Moderate', 'High'], yticklabels=['Low', 'Moderate', 'High'])
plt.title('Random Forest - Multimodal Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('Ground Truth Label')
plt.show()

## 7. Feature Importance Analysis (Explainability)

In [ ]:
feat_importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
plt.figure(figsize=(8, 4))
feat_importances.plot(kind='barh', color='#4F46E5')
plt.gca().invert_yaxis()
plt.title('Feature Importance for Level 2 ML Screener')
plt.xlabel('Relative Gini Importance')
plt.show()

print(feat_importances)

## 8. Export Model Artifacts for Deployment

In [ ]:
os.makedirs('model_artifacts', exist_ok=True)
joblib.dump(rf, 'model_artifacts/random_forest.joblib')
joblib.dump(lr, 'model_artifacts/logistic_regression.joblib')
joblib.dump(scaler, 'model_artifacts/scaler.joblib')

export_dict = {
    'features': FEATURE_COLS,
    'classes': ['Low', 'Moderate', 'High'],
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist(),
    'lr_coef': lr.coef_.tolist(),
    'lr_intercept': lr.intercept_.tolist(),
    'rf_feature_importances': feat_importances.to_dict()
}

with open('model_artifacts/model_bundle.json', 'w') as f:
    json.dump(export_dict, f, indent=2)

print("Model artifacts exported to model_artifacts/")